# Databricks Workflows — Diagram & Theory

Run the cell below first — it renders the job diagram as an actual notebook output (no external image file needed). Then read the theory sections below it.

In [0]:
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(10, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 7)
ax.axis("off")

def box(x, y, w, h, title, subtitle, color):
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.05,rounding_size=0.08",
                                 linewidth=1, edgecolor=color, facecolor=color, alpha=0.15))
    ax.text(x + w/2, y + h*0.62, title, ha="center", va="center", fontsize=11, fontweight="bold", color=color)
    ax.text(x + w/2, y + h*0.28, subtitle, ha="center", va="center", fontsize=9, color=color)

def arrow(x1, y1, x2, y2):
    ax.add_patch(FancyArrowPatch((x1, y1), (x2, y2), arrowstyle="-|>", mutation_scale=15,
                                  linewidth=1.2, color="#444444"))

# Trigger + parameters
box(0.3, 5.7, 2.8, 0.9, "Schedule trigger", "Daily at 8:00 AM", "#5F5E5A")
box(6.9, 5.7, 2.8, 0.9, "Job parameters", "region=AMER, run_date", "#5F5E5A")

# Tasks
box(0.3, 4.1, 2.8, 0.9, "ingest_data", "Task 1", "#0F6E56")
box(3.6, 4.1, 2.8, 0.9, "transform_data", "Task 2 - depends on Task 1", "#0F6E56")
box(3.6, 2.5, 2.8, 0.9, "validate", "Task 3 - depends on Task 2", "#0F6E56")

# Outcomes
box(0.3, 0.6, 2.8, 0.9, "Retry on failure", "Up to 2 retries", "#854F0B")
box(6.9, 0.6, 2.8, 0.9, "Notification", "Email/Slack on success or fail", "#993C1D")

arrow(1.7, 5.7, 1.7, 5.0)
arrow(8.3, 5.7, 5.0, 5.0)
arrow(3.1, 4.55, 3.6, 4.55)
arrow(5.0, 4.1, 5.0, 3.4)
arrow(4.5, 2.5, 1.7, 1.5)
arrow(5.5, 2.5, 8.3, 1.5)

plt.title("Multi-task Job: schedule -> tasks -> retry/notify", fontsize=12, pad=20)
plt.tight_layout()
plt.show()

## 1. Jobs & Multi-task Jobs

**Definition:** A Job is Databricks' managed way to run code (notebook, script, or SQL) without provisioning or babysitting a cluster yourself — Databricks spins up compute, runs it, tears it down, and keeps the logs. A **multi-task Job** breaks a pipeline into separate tasks linked by "depends on," so Databricks builds a DAG and only runs a task once everything it depends on has succeeded (as drawn above).

**Real-life example:** a nightly sales reporting pipeline as three tasks -
- `ingest_data` pulls yesterday's raw order files into a staging table
- `transform_data` (depends on task 1) cleans and aggregates into a summary table
- `validate` (depends on task 2) checks the summary table isn't empty or broken before a dashboard reads it

One giant notebook would mean re-running the expensive ingest step every time step 3 has a bug, and two engineers editing the same file. Splitting into tasks means each piece is owned, tested, and re-run independently — the same reasoning your team already applies to Airflow DAGs.

## 2. Scheduling & Parameters

**Scheduling:** a cron trigger or the simple UI picker that runs the Job automatically - daily, hourly, on file arrival, or purely manual "Run now".

**Parameters:** job-level key/value inputs passed into each task via `dbutils.widgets`, so the same Job definition works across contexts instead of being hardcoded per environment.

**Real example:** the sales pipeline is scheduled with the cron expression `0 0 6 * * ?` (6am daily) so the report is ready before stand-up. Instead of hardcoding `region = 'AMER'` inside the notebook, the Job exposes a `region` parameter - the exact same three tasks run for the EMEA or APAC team by changing one parameter value, with zero code duplication.

## 3. Notifications & Retry Logic

**Notifications:** email / Slack / webhook alerts on job or task start, success, or failure - pure configuration in the Job UI, no notebook code.

**Retry logic:** if a task fails, Databricks can automatically re-run just that task (not the whole pipeline) N times, optionally with a delay, before the Job is marked failed.

**Real example:** `ingest_data` occasionally hits a transient GCS write-lock from an upstream process - not a real bug. Setting 2 retries with a 5-minute gap lets it self-heal without paging anyone. Meanwhile `validate` has a Slack notification on failure only, so the team is pinged the moment a genuine data-quality issue slips through, not on every routine successful run.